# NPU 코어모드(single / global4 / global8) 추론 비교

같은 입력을 **코어모드만 바꿔** 돌려 단건 지연과 멀티채널 처리시간을 비교한다.

> **핵심: 모드를 바꾸려고 컴파일할 필요 없다.** Mobilint가 4종(single/multi/global4/global8)을
> 미리 컴파일해 HF(`PIA-SPACE-LAB/MXQ_NPU`)에 올려놨고, `MXQInferenceFull.from_hf(scheme=...)`
> (또는 `assets.ensure_full_mxq(scheme=...)`) **한 줄로 골라** 받는다. 출력(임베딩)은 4모드 동일,
> **속도/메모리만 다르다**(전부 원본 대비 cos 0.99).

| 모드 | 1장에 쓰는 코어 | 카드당 동시 슬롯 | 성격 |
|---|---|---|---|
| `single` | 1코어 | 8장 | 8코어 독립 → throughput |
| `global4` | 4코어 | 2장 | 4코어 분담, 채널 균형 최선 |
| `global8` | 8코어 | 1장 | 단건 latency 최소 |
| `multi` | 클러스터 협력 | — | 이 워크로드선 비효율(비권장) |

상세 실측: `../../reports/performance/NPU_coremode_benchmark.md`, `../../reports/performance/NPU_full_pipeline_e2e.md`

## 0. 준비 — NPU 감지 + 입력 1장

`pe_npu_host` 커널(qbruntime). NPU가 7대면 7대 전부에 분산, 1대면 1대로 동작(코어모드 비교는 동일하게 됨).

In [ ]:
import sys, os, glob, time
sys.path.insert(0, os.path.abspath("../.."))
import numpy as np
import pe_npu
from pe_npu import assets
import qbruntime

def detect_npus():
    ids = []
    for p in glob.glob("/dev/aries*"):
        s = os.path.basename(p)[len("aries"):]
        if s.isdigit(): ids.append(int(s))
    return sorted(ids)

NPUS = detect_npus()
assert NPUS, "NPU(/dev/aries*) 없음 — mobilint-cli status 확인"
print(f"장착 NPU: {NPUS} ({len(NPUS)}장, {len(NPUS)*8}코어)")

# 입력 1장 (HWC float32). 지연 측정용으로 동일 입력 재사용.
imgs = sorted(glob.glob("images/*.jpg"))
chw  = pe_npu.preprocess_image(imgs[0]) if imgs else np.random.randn(3,336,336).astype(np.float32)
X    = np.ascontiguousarray(np.asarray(chw).transpose(1,2,0).astype(np.float32))   # (336,336,3)
print("입력:", X.shape, X.dtype)

## 1. 가장 간단 — 모드 골라 추론 (`from_hf`)

이게 전부다. `scheme`만 바꾸면 된다.

In [ ]:
m = pe_npu.MXQInferenceFull.from_hf(scheme="global4")   # single | multi | global4 | global8
emb = m.infer(chw[None])                                 # (1, 1024)
print("core mode:", m.model.get_core_mode(), "| out:", emb.shape)
m.model.dispose()

## 2. 단건 지연 — 모드별 (1장, 1카드)

코어가 1장을 몇 개로 분담하느냐에 따라 단건 시간이 달라진다: global8(8코어) < global4(4코어) < single(1코어).

In [ ]:
MODES = ["single", "global4", "global8"]   # multi는 비권장이라 생략(원하면 추가)

def median_ms(fn, repeat=20):
    ts=[]
    for _ in range(repeat):
        t=time.perf_counter(); fn(); ts.append((time.perf_counter()-t)*1000)
    ts.sort(); return ts[len(ts)//2]

print(f"{'mode':8s} {'1장 지연(ms)':>12s}")
single_lat = {}
for mode in MODES:
    mxq = assets.ensure_full_mxq(scheme=mode)            # HF <mode>/pe_full.mxq 자동 다운
    mm  = qbruntime.Model(mxq); mm.launch(qbruntime.Accelerator(NPUS[0]))
    mm.infer(X)                                          # warmup
    dt  = median_ms(lambda: mm.infer(X))
    single_lat[mode] = dt
    print(f"{mode:8s} {dt:12.1f}")
    mm.dispose()

## 3. 멀티채널 처리시간 — 모드별 (N장 동시, 전 카드 async 분산)

N채널을 장착 카드에 **라운드로빈 + `infer_async`**로 던지고 전부 끝나는 시간을 잰다.
디스패치 코드는 모드 무관 **동일**하다 — MXQ에 박힌 코어모드만 다름.

- `single`: 카드당 8슬롯(8코어 독립) → 고채널 throughput
- `global4`: 카드당 2슬롯(4코어×2) → 채널 균형 최선
- `global8`: 카드당 1슬롯(8코어) → 단건 최소, 저채널 유리

In [ ]:
# 채널 스윕 지점 (카드 수에 맞춰 자동). 7대면 [1,7,14,28,56].
ncores = len(NPUS)*8
CHANNELS = sorted(set([1, len(NPUS), len(NPUS)*2, ncores//2, ncores]))
print("채널 스윕:", CHANNELS)

def bench_mode_sweep(mode, channels, repeat=3):
    mxq = assets.ensure_full_mxq(scheme=mode)
    models=[]
    for d in NPUS:
        cfg = qbruntime.ModelConfig(); cfg.set_async_pipeline_enabled(True)   # 8코어 async
        mm  = qbruntime.Model(mxq, cfg); mm.launch(qbruntime.Accelerator(d)); models.append(mm)
    for mm in models: mm.infer_async(X).get()             # warmup
    out={}
    for n in channels:
        def run():
            futs=[models[i % len(models)].infer_async(X) for i in range(n)]   # 라운드로빈
            for f in futs: f.get()
        out[n]=median_ms(run, repeat)
    for mm in models: mm.dispose()
    return out

results = {mode: bench_mode_sweep(mode, CHANNELS) for mode in MODES}

# 표 출력
hdr = "ch     " + "".join(f"{m:>10s}" for m in MODES)
print(hdr); print("-"*len(hdr))
for n in CHANNELS:
    row = f"{n:<7d}" + "".join(f"{results[m][n]:10.1f}" for m in MODES)
    print(row)
print("\n(단위 ms, median, 전 카드 async 분산)")

## 4. 그래프

In [ ]:
import matplotlib
matplotlib.use("Agg") if False else None
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7,4))
for mode in MODES:
    ax.plot(CHANNELS, [results[mode][n] for n in CHANNELS], marker="o", label=mode)
ax.set_xlabel("channels (images in batch)"); ax.set_ylabel("batch latency (ms)")
ax.set_title(f"Core mode vs channels ({len(NPUS)} ARIES, async round-robin)")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 5. 모드 선택 가이드 (full NPU, 실측 기반)

| 상황 | 권장 | 근거 |
|---|---|---|
| 단건/저채널 실시간(≤카드수) | **global8** | 단건 ~71ms 최저 |
| 중간 채널(8~16) | **global4** | global8이 카드당 1슬롯이라 2배 튐, global4가 평탄 |
| 고채널 throughput(≥28) | **global4** 또는 **single** | 7카드 56ch서 global4 ~488ms 최선 |
| 메모리 빠듯 | **global8**(341MB/card) / global4(387) | single 582로 가장 큼 |
| `multi` | 비권장 | 이 워크로드선 가장 느림 |

> 출력(임베딩)은 4모드 동일 — 정확도/결과는 안 바뀌고 **스케줄링(속도·메모리)만** 바뀐다.
> 운영 코드에선 `MXQInferenceFull.from_hf(scheme="...")` 한 줄로 모드 고정, 나머지는 그대로.
> 컴파일해 직접 만들려면: `python -m pe_npu.compile --qk16 --scheme <mode> ...` (`../../reports/performance/NPU_coremode_benchmark.md`).